# 01 Load and QC Datalogger Data

This notebook loads ESP8266 datalogger measurement CSV files from **2026-05-29 onward only**. It preserves duplicated sensor columns from the two I2C buses by renaming repeated headers with `__bus0`, `__bus1`, ... suffixes.

In [1]:
from pathlib import Path
import csv
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

DATA_START_DATE = "20260529"
EXPECTED_INTERVAL_SECONDS = 20
GPS_STALE_THRESHOLD_MS = 120000
DATA_DIR = Path("..") / "test_data"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

print("pandas", pd.__version__)
print("data directory", DATA_DIR.resolve())

pandas 3.0.3
data directory C:\Users\jonmuell\Documents\GitHub\datalogger_esp8266\test_data


In [2]:
def data_file_date(path: Path) -> str:
    match = re.fullmatch(r"data_(\d{8})\.csv", path.name)
    return match.group(1) if match else ""


def measurement_files(data_dir: Path = DATA_DIR, start_date: str = DATA_START_DATE) -> list[Path]:
    files = []
    for path in sorted(data_dir.glob("data_*.csv")):
        date = data_file_date(path)
        if date and date >= start_date:
            files.append(path)
    return files


def unique_headers(headers: list[str]) -> list[str]:
    total = Counter(headers)
    seen = defaultdict(int)
    out = []
    for header in headers:
        if total[header] == 1:
            out.append(header)
        else:
            bus = seen[header]
            out.append(f"{header}__bus{bus}")
            seen[header] += 1
    return out


def inspect_row_widths(path: Path) -> tuple[list[str], pd.DataFrame]:
    issues = []
    with path.open("r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        raw_headers = next(reader)
        expected_width = len(raw_headers)
        for line_number, row in enumerate(reader, start=2):
            if len(row) != expected_width:
                issues.append({
                    "source_file": path.name,
                    "line_number": line_number,
                    "expected_columns": expected_width,
                    "actual_columns": len(row),
                })
    return raw_headers, pd.DataFrame(issues)


def read_measurement_file(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw_headers, width_issues = inspect_row_widths(path)
    names = unique_headers(raw_headers)
    df = pd.read_csv(
        path,
        names=names,
        skiprows=1,
        na_values=["", "nan", "NaN", "NAN"],
        keep_default_na=True,
        on_bad_lines="skip",
    )
    df.insert(0, "source_file", path.name)
    df.insert(1, "source_date", data_file_date(path))
    return df, width_issues


files = measurement_files()
assert files, "No measurement files found from the configured start date."
print("Selected measurement files:")
for file in files:
    print(" -", file.name)

assert all(data_file_date(file) >= DATA_START_DATE for file in files)

Selected measurement files:
 - data_20260529.csv
 - data_20260530.csv
 - data_20260531.csv
 - data_20260601.csv
 - data_20260602.csv
 - data_20260603.csv
 - data_20260604.csv
 - data_20260607.csv
 - data_20260611.csv
 - data_20260612.csv
 - data_20260613.csv
 - data_20260614.csv
 - data_20260615.csv
 - data_20260616.csv
 - data_20260617.csv
 - data_20260618.csv


In [3]:
frames = []
width_issue_frames = []
for file in files:
    frame, issues = read_measurement_file(file)
    frames.append(frame)
    width_issue_frames.append(issues)

data = pd.concat(frames, ignore_index=True)
row_width_issues = pd.concat(width_issue_frames, ignore_index=True) if width_issue_frames else pd.DataFrame()

data["timestamp_utc"] = pd.to_datetime(data["timestamp_utc"], errors="coerce", utc=False)
if "timestamp_calc_utc" in data.columns:
    data["timestamp_calc_utc"] = pd.to_datetime(data["timestamp_calc_utc"], errors="coerce", utc=False)
else:
    data["timestamp_calc_utc"] = pd.NaT
if "gps_time_fresh" in data.columns:
    data["gps_time_fresh_bool"] = pd.to_numeric(data["gps_time_fresh"], errors="coerce").fillna(1).astype(bool)
else:
    data["gps_time_fresh_bool"] = True
gps_age = pd.to_numeric(data["gps_age_ms"], errors="coerce") if "gps_age_ms" in data.columns else pd.Series(pd.NA, index=data.index, dtype="Float64")
data["gps_age_too_high"] = gps_age > GPS_STALE_THRESHOLD_MS
data["gps_time_stale"] = (~data["gps_time_fresh_bool"]) | data["gps_age_too_high"].fillna(False)
if "timestamp_calc_source" in data.columns:
    calc_source = data["timestamp_calc_source"].astype("string").str.lower()
else:
    calc_source = pd.Series(pd.NA, index=data.index, dtype="string")
if "timestamp_boot_ms" in data.columns:
    boot_ms = pd.to_numeric(data["timestamp_boot_ms"], errors="coerce")
elif "uptime_ms" in data.columns:
    boot_ms = pd.to_numeric(data["uptime_ms"], errors="coerce")
else:
    boot_ms = pd.Series(pd.NA, index=data.index, dtype="Float64")
data["analysis_time"] = data["timestamp_utc"]
groups = data.groupby("boot_id", dropna=False) if "boot_id" in data.columns else [(None, data)]
for _, group in groups:
    fresh = (~group["gps_time_stale"]) & group["timestamp_utc"].notna() & boot_ms.loc[group.index].notna()
    if not fresh.any():
        continue
    base_idx = fresh[fresh].index[-1]
    base_utc = data.at[base_idx, "timestamp_utc"]
    base_boot = boot_ms.loc[base_idx]
    if pd.isna(base_boot):
        continue
    stale_idx = group.index[group["gps_time_stale"] & boot_ms.loc[group.index].notna()]
    data.loc[stale_idx, "analysis_time"] = base_utc + pd.to_timedelta(boot_ms.loc[stale_idx] - base_boot, unit="ms")
data["analysis_time"] = data["timestamp_calc_utc"].combine_first(data["analysis_time"])
data["time_corrected_from_uptime"] = (calc_source == "uptime").fillna(False) | (
    data["gps_time_stale"].fillna(False) & data["analysis_time"].notna() & data["timestamp_utc"].notna() & (data["analysis_time"] != data["timestamp_utc"])
)
data["file_date"] = pd.to_datetime(data["source_date"], format="%Y%m%d", errors="coerce").dt.date
data["date"] = data["analysis_time"].dt.date
data["hour"] = data["analysis_time"].dt.hour

print("Rows loaded:", len(data))
print("Columns loaded:", len(data.columns))
print("Timestamp parse failures:", int(data["timestamp_utc"].isna().sum()))
print("Rows with stale GPS time:", int(data["gps_time_stale"].sum()))
print("Rows corrected from uptime:", int(data["time_corrected_from_uptime"].sum()))
print("Row width issues:", len(row_width_issues))

data.head()

Rows loaded: 68202
Columns loaded: 92
Timestamp parse failures: 12811
Rows with stale GPS time: 37852
Rows corrected from uptime: 37852
Row width issues: 3191


,source_file,source_date,timestamp_utc,lat,lon,alt,nb_sat,HDOP,bat.mV,bat.perc,bme_T.C__bus0,bme_RH.perc__bus0,bme_P.Pa__bus0,bme_H2O.mmol_mol__bus0,bme_T.C.cal__bus0,bme_T.flag__bus0,bme_P.Pa.cal__bus0,bme_P.flag__bus0,bme_H2O.mmol_mol.cal__bus0,bme_H2O.mmol_mol.flag__bus0,bme_RH.perc.cal__bus0,bme_RH.flag__bus0,scd_T.C__bus0,scd_RH.perc__bus0,scd_CO2.ppm__bus0,scd_CO2.ppm.cal__bus0,scd_CO2.flag__bus0,sen_T.C__bus0,sen_O2.mmol_mol__bus0,sen_O2.mmol_mol.cal__bus0,sen_O2.flag__bus0,bme_T.C__bus1,bme_RH.perc__bus1,bme_P.Pa__bus1,bme_H2O.mmol_mol__bus1,bme_T.C.cal__bus1,bme_T.flag__bus1,bme_P.Pa.cal__bus1,bme_P.flag__bus1,bme_H2O.mmol_mol.cal__bus1,bme_H2O.mmol_mol.flag__bus1,bme_RH.perc.cal__bus1,bme_RH.flag__bus1,scd_T.C__bus1,scd_RH.perc__bus1,scd_CO2.ppm__bus1,scd_CO2.ppm.cal__bus1,scd_CO2.flag__bus1,sen_T.C__bus1,sen_O2.mmol_mol__bus1,sen_O2.mmol_mol.cal__bus1,sen_O2.flag__bus1,CRC16,boot_id,sample_counter,uptime_ms,timestamp_boot_ms,reset_reason,gps_date_valid,gps_location_valid,gps_chars_processed,gps_age_ms,free_heap,max_heap_block,min_heap_block,i2c_slow_count,i2c_error_count,write_status,gps_time_fresh,gps_location_fresh,gps_location_age_ms,gps_stale_count,gps_recovery_count,timestamp_calc_utc,timestamp_calc_source,sen_raw_V__bus0,sen_raw_V__bus1,i2c_recovery_count,i2c_bus0_error_count,i2c_bus0_consecutive_error_count,i2c_bus0_recovery_count,i2c_bus1_error_count,i2c_bus1_consecutive_error_count,i2c_bus1_recovery_count,gps_time_fresh_bool,gps_age_too_high,gps_time_stale,analysis_time,time_corrected_from_uptime,file_date,date,hour
0,data_20260529.csv,20260529,2026-05-29 09:56:14,47.377724,8.548743,466.99,9,1.37,NaN,NaN,29.97,26.56,97504.46,11.50,29.97,-1,97504.46,-1,11.50,-1,26.56,-3.0,35.56,26.78,440.0,440.0,-1,35.44,123.0,123.0,-1,30.51,30.38,97649.61,13.56,30.51,-1,97649.61,-1,13.56,-1,30.38,-3.0,35.71,37.29,651.0,609.09,0,34.76,209.0,209.0,-1,C9DB,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,False,2026-05-29 09:56:14,False,2026-05-29,2026-05-29,9
1,data_20260529.csv,20260529,2026-05-29 09:56:36,47.377730,8.548743,467.02,10,1.06,NaN,NaN,29.84,26.26,97503.51,11.29,29.84,-1,97503.51,-1,11.29,-1,26.26,-3.0,35.34,26.88,435.0,435.0,-1,35.21,124.0,124.0,-1,30.44,30.85,97650.55,13.71,30.44,-1,97650.55,-1,13.71,-1,30.85,-3.0,35.65,37.43,654.0,611.90,0,34.87,210.0,210.0,-1,D9A2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,False,2026-05-29 09:56:36,False,2026-05-29,2026-05-29,9
2,data_20260529.csv,20260529,2026-05-29 09:56:58,47.377733,8.548745,466.88,11,1.04,NaN,NaN,29.73,26.08,97503.09,11.14,29.73,-1,97503.09,-1,11.14,-1,26.08,-3.0,35.21,26.69,412.0,412.0,-1,35.21,125.0,125.0,-1,30.36,31.19,97650.68,13.80,30.36,-1,97650.68,-1,13.80,-1,31.19,-3.0,35.57,37.82,656.0,613.77,0,34.76,210.0,210.0,-1,DBD5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,False,2026-05-29 09:56:58,False,2026-05-29,2026-05-29,9
3,data_20260529.csv,20260529,2026-05-29 09:59:38,47.377714,8.548743,465.73,9,1.68,NaN,NaN,29.14,27.16,97498.95,11.22,29.14,-1,97498.95,-1,11.22,-1,27.16,-3.0,34.71,27.75,409.0,409.0,-1,34.76,127.0,127.0,-1,29.69,35.56,97652.70,15.14,29.69,-1,97652.70,-1,15.14,-1,35.56,-3.0,34.92,41.97,706.0,660.57,0,34.76,210.0,210.0,-1,9D1E,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,False,2026-05-29 09:59:38,False,2026-05-29,2026-05-29,9
4,data_20260529.csv,20260529,2026-05-29 09:59:59,47.377713,8.548750,465.69,9,1.31,NaN,NaN,29.08,27.29,97498.00,11.23,29.08,-1,97498.00,-1,11.23,-1,27.29,-3.0,34.59,27.79,405.0,405.0,-1,34.76,128.0,128.0,-1,29.68,35.83,97652.09,15.24,29.68,-1,97652.09,-1,15.24,-1,35.83,-3.0,34.96,42.00,708.0,662.44,0,34.76,210.0,210.0,-1,5AEC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [4]:
def compute_gaps(df: pd.DataFrame, expected_interval_s: int = EXPECTED_INTERVAL_SECONDS) -> pd.DataFrame:
    pieces = []
    valid = df.dropna(subset=["analysis_time"]).sort_values(["source_file", "analysis_time"])
    for source_file, group in valid.groupby("source_file", sort=True):
        ts = group["analysis_time"].reset_index(drop=True)
        deltas = ts.diff().dt.total_seconds()
        gap = pd.DataFrame({
            "source_file": source_file,
            "previous_timestamp": ts.shift(1),
            "analysis_time": ts,
            "delta_seconds": deltas,
        })
        pieces.append(gap)
    out = pd.concat(pieces, ignore_index=True)
    out["missing_samples_est"] = np.maximum(
        0,
        np.floor(out["delta_seconds"].fillna(expected_interval_s) / expected_interval_s).astype(int) - 1,
    )
    return out


gaps_all = compute_gaps(data)
large_gaps = gaps_all[gaps_all["delta_seconds"] > 120].copy()
large_gaps["gap_minutes"] = large_gaps["delta_seconds"] / 60

duplicate_timestamps = (
    data.dropna(subset=["analysis_time"])
    .groupby(["source_file", "analysis_time"])
    .size()
    .reset_index(name="count")
    .query("count > 1")
)

file_summary = (
    data.groupby("source_file", as_index=False)
    .agg(
        rows=("analysis_time", "size"),
        parseable_timestamps=("analysis_time", "count"),
        start_analysis_time=("analysis_time", "min"),
        end_analysis_time=("analysis_time", "max"),
        gps_sat_median=("nb_sat", "median"),
        hdop_median=("HDOP", "median"),
    )
)
gap_summary = large_gaps.groupby("source_file", as_index=False).agg(
    gaps_gt_120s=("delta_seconds", "size"),
    max_gap_s=("delta_seconds", "max"),
    missing_samples_est=("missing_samples_est", "sum"),
)
file_summary = file_summary.merge(gap_summary, on="source_file", how="left").fillna({
    "gaps_gt_120s": 0,
    "max_gap_s": 0,
    "missing_samples_est": 0,
})

file_summary

,source_file,rows,parseable_timestamps,start_analysis_time,end_analysis_time,gps_sat_median,hdop_median,gaps_gt_120s,max_gap_s,missing_samples_est
0,data_20260529.csv,1751,1751,2026-05-29 09:56:14,2026-05-29 17:38:49.000,13.0,0.860,4.0,17068.000,1161.0
1,data_20260530.csv,2300,2300,2026-05-30 06:31:05,2026-05-30 15:59:36.000,13.0,0.750,0.0,0.000,0.0
2,data_20260531.csv,1975,1975,2026-05-31 07:31:09,2026-05-31 10:09:26.000,14.0,0.710,0.0,0.000,0.0
3,data_20260601.csv,2784,2784,2026-06-01 03:05:40,2026-06-01 15:46:53.000,15.0,0.680,1.0,40596.000,2028.0
4,data_20260602.csv,2773,2773,2026-06-02 07:13:57,2026-06-02 23:59:53.000,14.0,0.780,0.0,0.000,0.0
5,data_20260603.csv,2294,2294,2026-06-03 00:00:16,2026-06-03 13:52:51.000,14.0,0.800,3.0,16005.000,883.0
6,data_20260604.csv,15219,15219,2026-06-04 00:00:16,2026-06-07 14:28:16.807,14.0,0.750,0.0,0.000,0.0
7,data_20260607.csv,13082,13082,2026-06-07 14:30:16,2026-06-11 13:25:21.292,14.0,0.680,3.0,166.079,18.0
8,data_20260611.csv,2,2,2026-06-11 14:17:05,2026-06-11 14:17:27.000,10.0,0.875,0.0,0.000,0.0
9,data_20260612.csv,3925,3925,2026-06-12 00:00:07,2026-06-12 23:59:47.000,14.0,0.760,0.0,0.000,0.0


In [5]:
print("Large gaps over 120 seconds:")
display(large_gaps[["source_file", "previous_timestamp", "analysis_time", "delta_seconds", "gap_minutes", "missing_samples_est"]])

print("Duplicate timestamps:", len(duplicate_timestamps))
display(duplicate_timestamps.head(20))

print("Row width issues:", len(row_width_issues))
display(row_width_issues.head(20))

Large gaps over 120 seconds:


,source_file,previous_timestamp,analysis_time,delta_seconds,gap_minutes,missing_samples_est
3,data_20260529.csv,2026-05-29 09:56:58.000,2026-05-29 09:59:38.000,160.000,2.666667,7
450,data_20260529.csv,2026-05-29 11:03:36.000,2026-05-29 12:36:07.000,5551.000,92.516667,276
1223,data_20260529.csv,2026-05-29 12:36:07.000,2026-05-29 17:20:35.000,17068.000,284.466667,852
1249,data_20260529.csv,2026-05-29 17:20:35.000,2026-05-29 17:29:44.000,549.000,9.150000,26
8005,data_20260601.csv,2026-06-01 03:05:40.000,2026-06-01 14:22:16.000,40596.000,676.600000,2028
12003,data_20260603.csv,2026-06-03 02:20:15.000,2026-06-03 02:37:53.000,1058.000,17.633333,51
13090,data_20260603.csv,2026-06-03 09:12:11.000,2026-06-03 09:23:32.000,681.000,11.350000,33
13873,data_20260603.csv,2026-06-03 09:25:01.000,2026-06-03 13:51:46.000,16005.000,266.750000,799
29200,data_20260607.csv,2026-06-07 15:08:05.000,2026-06-07 15:10:34.576,149.576,2.492933,6
29201,data_20260607.csv,2026-06-07 15:10:34.576,2026-06-07 15:13:20.655,166.079,2.767983,7


Duplicate timestamps: 14


,source_file,analysis_time,count
180,data_20260529.csv,2026-05-29 11:03:36,270
181,data_20260529.csv,2026-05-29 12:36:07,773
182,data_20260529.csv,2026-05-29 17:20:35,26
208,data_20260529.csv,2026-05-29 17:38:49,477
1775,data_20260530.csv,2026-05-30 15:59:36,734
2213,data_20260531.csv,2026-05-31 10:09:26,1538
2214,data_20260601.csv,2026-06-01 03:05:40,1979
2448,data_20260601.csv,2026-06-01 15:46:53,572
5608,data_20260603.csv,2026-06-03 02:20:15,34
6701,data_20260603.csv,2026-06-03 09:25:01,778


Row width issues: 3191


,source_file,line_number,expected_columns,actual_columns
0,data_20260603.csv,2296,51,66
1,data_20260603.csv,2297,51,66
2,data_20260603.csv,2298,51,66
3,data_20260603.csv,2299,51,66
4,data_20260603.csv,2300,51,66
5,data_20260603.csv,2301,51,66
6,data_20260603.csv,2302,51,66
7,data_20260603.csv,2303,51,66
8,data_20260603.csv,2304,51,66
9,data_20260603.csv,2305,51,66


In [6]:
hourly_base = data.dropna(subset=["analysis_time"]).assign(
    date=lambda x: x["analysis_time"].dt.strftime("%Y-%m-%d"),
    hour=lambda x: x["analysis_time"].dt.hour,
)
hourly_coverage = hourly_base.groupby(["date", "hour"], as_index=False).agg(
    rows=("analysis_time", "size"),
    unique_timestamps=("analysis_time", "nunique"),
    stale_gps_rows=("gps_time_stale", "sum"),
    corrected_from_uptime_rows=("time_corrected_from_uptime", "sum"),
    max_gps_age_ms=("gps_age_ms", lambda x: pd.to_numeric(x, errors="coerce").max()),
)
hourly_coverage["duplicate_timestamp_rows"] = hourly_coverage["rows"] - hourly_coverage["unique_timestamps"]
hourly_coverage["expected_rows"] = 3600 / EXPECTED_INTERVAL_SECONDS
hourly_coverage["coverage_fraction"] = (hourly_coverage["unique_timestamps"] / hourly_coverage["expected_rows"]).clip(upper=1)
hourly_coverage["stale_gps_fraction"] = hourly_coverage["stale_gps_rows"] / hourly_coverage["rows"]
hourly_coverage["has_stale_gps"] = hourly_coverage["stale_gps_rows"] > 0

display(hourly_coverage.head(48))
display(hourly_coverage.query("date == '2026-05-31'"))

,date,hour,rows,unique_timestamps,stale_gps_rows,corrected_from_uptime_rows,max_gps_age_ms,duplicate_timestamp_rows,expected_rows,coverage_fraction,stale_gps_fraction,has_stale_gps
0,2026-05-29,9,5,5,0,0,NaN,0,180.0,0.027778,0.0,False
1,2026-05-29,10,165,165,0,0,NaN,0,180.0,0.916667,0.0,False
2,2026-05-29,11,280,11,0,0,NaN,269,180.0,0.061111,0.0,False
3,2026-05-29,12,773,1,0,0,NaN,772,180.0,0.005556,0.0,False
4,2026-05-29,17,528,27,0,0,NaN,501,180.0,0.150000,0.0,False
5,2026-05-30,6,80,80,0,0,NaN,0,180.0,0.444444,0.0,False
6,2026-05-30,7,165,165,0,0,NaN,0,180.0,0.916667,0.0,False
7,2026-05-30,8,165,165,0,0,NaN,0,180.0,0.916667,0.0,False
8,2026-05-30,9,166,166,0,0,NaN,0,180.0,0.922222,0.0,False
9,2026-05-30,10,165,165,0,0,NaN,0,180.0,0.916667,0.0,False


,date,hour,rows,unique_timestamps,stale_gps_rows,corrected_from_uptime_rows,max_gps_age_ms,duplicate_timestamp_rows,expected_rows,coverage_fraction,stale_gps_fraction,has_stale_gps
15,2026-05-31,7,80,80,0,0,NaN,0,180.0,0.444444,0.0,False
16,2026-05-31,8,165,165,0,0,NaN,0,180.0,0.916667,0.0,False
17,2026-05-31,9,166,166,0,0,NaN,0,180.0,0.922222,0.0,False
18,2026-05-31,10,1564,27,0,0,NaN,1537,180.0,0.150000,0.0,False


In [7]:
def status_file_date(path: Path) -> str:
    match = re.fullmatch(r"status_(\d{8})\.csv", path.name)
    return match.group(1) if match else ""


def read_status_file(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, na_values=["", "nan", "NaN", "NAN"], keep_default_na=True, on_bad_lines="skip")
    frame.insert(0, "source_file", path.name)
    frame.insert(1, "source_date", status_file_date(path))
    return frame


status_files = sorted(path for path in DATA_DIR.glob("status_*.csv") if status_file_date(path))
if status_files:
    status_data = pd.concat([read_status_file(path) for path in status_files], ignore_index=True, sort=False)
    print("Status rows loaded:", len(status_data))
    display(status_data.head(50))
else:
    status_data = pd.DataFrame()
    print("No status_YYYYMMDD.csv files found yet; this is expected for pre-patch data.")

new_diag_cols = [
    "boot_id", "sample_counter", "uptime_ms", "timestamp_boot_ms", "reset_reason",
    "timestamp_calc_utc", "timestamp_calc_source",
    "gps_date_valid", "gps_time_fresh", "gps_location_valid", "gps_location_fresh",
    "gps_chars_processed", "gps_age_ms", "gps_location_age_ms", "gps_stale_count", "gps_recovery_count",
    "free_heap", "max_heap_block", "min_heap_block", "i2c_slow_count",
    "i2c_error_count", "i2c_recovery_count", "write_status",
]
present_diag_cols = [col for col in new_diag_cols if col in data.columns]
print("New firmware diagnostic columns present:", present_diag_cols)
if present_diag_cols:
    display(data[present_diag_cols].head())

ParserError: Error tokenizing data. C error: Expected 18 fields in line 3, saw 24


In [ ]:
# Robust status loader and GPS/I2C fault classification for June 11+ data.
def status_file_date(path: Path) -> str:
    match = re.fullmatch(r"status_(\d{8})\.csv", path.name)
    return match.group(1) if match else ""


def read_status_file(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, na_values=["", "nan", "NaN", "NAN"], keep_default_na=True, on_bad_lines="skip")
    frame.insert(0, "source_file", path.name)
    frame.insert(1, "source_date", status_file_date(path))
    return frame


def real_sensor_columns(df: pd.DataFrame) -> list[str]:
    prefixes = ("bme_", "scd_", "sen_", "Sin.", "Sout.", "mlx_")
    blocked = ("flag", ".cal")
    return [col for col in df.columns if col.startswith(prefixes) and not any(token in col for token in blocked)]


status_files = sorted(path for path in DATA_DIR.glob("status_*.csv") if status_file_date(path) >= "20260611")
if status_files:
    status_data = pd.concat([read_status_file(path) for path in status_files], ignore_index=True, sort=False)
    for col in status_data.columns:
        if col not in {"source_file", "source_date", "event", "reset_reason", "CRC16"}:
            status_data[col] = pd.to_numeric(status_data[col], errors="coerce")
    print("Status rows loaded from 2026-06-11 onward:", len(status_data))
    display(status_data.groupby(["source_file", "event"], dropna=False).size().rename("rows").reset_index())
else:
    status_data = pd.DataFrame()
    print("No status_YYYYMMDD.csv files found from 2026-06-11 onward.")

sensor_cols = real_sensor_columns(data)
for col in sensor_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")
if "valid_sensor_value_count" not in data.columns:
    data["valid_sensor_value_count"] = data[sensor_cols].notna().sum(axis=1) if sensor_cols else 0
if "missing_sensor_value_count" not in data.columns:
    data["missing_sensor_value_count"] = data[sensor_cols].isna().sum(axis=1) if sensor_cols else 0

gps_chars = pd.to_numeric(data.get("gps_chars_processed", pd.Series(pd.NA, index=data.index)), errors="coerce")
gps_age = pd.to_numeric(data.get("gps_age_ms", pd.Series(pd.NA, index=data.index)), errors="coerce")
valid_sensor_values = pd.to_numeric(data["valid_sensor_value_count"], errors="coerce").fillna(0)
quarantine = pd.to_numeric(data.get("gps_quarantine_active", pd.Series(0, index=data.index)), errors="coerce").fillna(0).astype(bool)

classified = data.dropna(subset=["analysis_time"]).copy()
classified["gps_chars_zero"] = gps_chars.loc[classified.index].fillna(1).eq(0)
classified["gps_age_too_high"] = gps_age.loc[classified.index].gt(GPS_STALE_THRESHOLD_MS).fillna(False)
classified["sensor_data_lost"] = valid_sensor_values.loc[classified.index].eq(0)
classified["gps_i2c_wedge_suspected"] = classified["sensor_data_lost"] & (
    classified["gps_chars_zero"] | classified["gps_age_too_high"] | quarantine.loc[classified.index]
)
classified["fault_class"] = np.select(
    [
        classified["gps_i2c_wedge_suspected"],
        classified["sensor_data_lost"],
        classified["gps_age_too_high"] | classified["gps_chars_zero"],
    ],
    ["gps_i2c_wedge_suspected", "sensor_data_lost", "gps_stale_only"],
    default="normal",
)

fault_summary = (
    classified.assign(date=lambda x: x["analysis_time"].dt.strftime("%Y-%m-%d"))
    .groupby(["date", "fault_class"], as_index=False)
    .agg(
        rows=("fault_class", "size"),
        first_time=("analysis_time", "min"),
        last_time=("analysis_time", "max"),
        median_valid_sensor_values=("valid_sensor_value_count", "median"),
        max_gps_age_ms=("gps_age_ms", "max"),
    )
)
print("GPS/I2C fault classification summary:")
display(fault_summary)

transitions = classified[[
    "analysis_time", "fault_class", "valid_sensor_value_count",
    "gps_chars_processed", "gps_age_ms", "source_file",
]].copy()
transitions["group"] = transitions["fault_class"].ne(transitions["fault_class"].shift()).cumsum()
fault_intervals = (
    transitions.groupby("group", as_index=False)
    .agg(
        fault_class=("fault_class", "first"),
        start=("analysis_time", "min"),
        end=("analysis_time", "max"),
        rows=("fault_class", "size"),
        min_valid_sensor_values=("valid_sensor_value_count", "min"),
        max_gps_age_ms=("gps_age_ms", "max"),
        first_source_file=("source_file", "first"),
        last_source_file=("source_file", "last"),
    )
)
fault_intervals["duration_minutes"] = (fault_intervals["end"] - fault_intervals["start"]).dt.total_seconds() / 60
fault_intervals = fault_intervals[fault_intervals["fault_class"] != "normal"]
print("Fault intervals:")
display(fault_intervals.sort_values("start"))
